# NaSch — Animated Road Evolution

Watch the Nagel-Schreckenberg road update in real time.
Cars are coloured by velocity (blue = slow, red = fast).

| Animation | What to watch |
|---|---|
| 1 | Free flow vs congestion side by side |
| 2 | Stop-and-go waves forming and propagating |
| 3 | Traffic lights: queue discharge in action |

**How to play:** run all cells — each produces an inline HTML5 animation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams.update({'figure.dpi': 100})

L         = 200
V_MAX     = 5
P_RAND    = 0.3
T_WARMUP  = 300
T_FRAMES  = 150
INTERVAL  = 60    # ms per frame


class NaSchOpen:
    '''
    NaSch CA with open boundary — cars enter at cell 0 with p_in.
    road[i] = velocity of car at cell i, or -1 if empty.
    '''
    def __init__(self, L=200, v_max=5, p_rand=0.3, p_in=0.5, seed=None):
        self.L      = L
        self.v_max  = v_max
        self.p_rand = p_rand
        self.p_in   = p_in
        self.rng    = np.random.default_rng(seed)
        self.road   = np.full(L, -1, dtype=int)

    def step(self):
        road, L, v_max, p_rand = self.road, self.L, self.v_max, self.p_rand
        car_pos = np.where(road >= 0)[0]
        if len(car_pos) == 0:
            self._inflow()
            return
        velocities = road[car_pos].copy()
        velocities = np.minimum(velocities + 1, v_max)
        # gaps
        gaps = np.full(len(car_pos), L, dtype=int)
        for i, p in enumerate(car_pos):
            idx = np.searchsorted(car_pos, p, side='right')
            if idx < len(car_pos):
                gaps[i] = car_pos[idx] - p - 1
            else:
                gaps[i] = L - p
        velocities = np.minimum(velocities, gaps)
        velocities = np.maximum(velocities, 0)
        rand_mask = self.rng.random(len(car_pos)) < p_rand
        velocities[rand_mask] = np.maximum(velocities[rand_mask] - 1, 0)
        new_road = np.full(L, -1, dtype=int)
        new_pos  = car_pos + velocities
        staying  = new_pos < L
        new_road[new_pos[staying]] = velocities[staying]
        self.road = new_road
        self._inflow()

    def _inflow(self):
        if self.road[0] < 0 and self.rng.random() < self.p_in:
            self.road[0] = 0

    def velocity_grid(self):
        '''Return float array: NaN=empty, else velocity normalised to [0,1].'''
        g = np.full(self.L, np.nan)
        mask = self.road >= 0
        g[mask] = self.road[mask] / self.v_max
        return g


class NaSchWithLights(NaSchOpen):
    def __init__(self, N=2, T_cycle=40, T_green=20, offset=False, **kwargs):
        super().__init__(**kwargs)
        self.N        = N
        self.T_cycle  = T_cycle
        self.T_green  = T_green
        self.offset   = offset
        self.t        = 0
        self.light_pos = np.array([L * (i+1) // (N+1) for i in range(N)], dtype=int)
        if offset:
            self.phase_offsets = np.array([int(i * T_cycle / N) for i in range(N)], dtype=int)
        else:
            self.phase_offsets = np.zeros(N, dtype=int)

    def _red_positions(self):
        red = [self.light_pos[i] for i in range(self.N)
               if (self.t + self.phase_offsets[i]) % self.T_cycle >= self.T_green]
        return np.array(red, dtype=int)

    def step(self):
        road, L, v_max, p_rand = self.road, self.L, self.v_max, self.p_rand
        car_pos = np.where(road >= 0)[0]
        if len(car_pos) == 0:
            self._inflow(); self.t += 1; return
        red_pos   = self._red_positions()
        obstacles = np.sort(np.concatenate([car_pos, red_pos]))
        velocities = road[car_pos].copy()
        velocities = np.minimum(velocities + 1, v_max)
        gaps = np.full(len(car_pos), L, dtype=int)
        for i, p in enumerate(car_pos):
            idx = np.searchsorted(obstacles, p, side='right')
            if idx < len(obstacles):
                gaps[i] = obstacles[idx] - p - 1
            else:
                gaps[i] = L - p
        velocities = np.minimum(velocities, gaps)
        velocities = np.maximum(velocities, 0)
        rand_mask = self.rng.random(len(car_pos)) < p_rand
        velocities[rand_mask] = np.maximum(velocities[rand_mask] - 1, 0)
        new_road = np.full(L, -1, dtype=int)
        new_pos  = car_pos + velocities
        staying  = new_pos < L
        new_road[new_pos[staying]] = velocities[staying]
        self.road = new_road
        self._inflow()
        self.t += 1

    def light_states(self):
        return {self.light_pos[i]:
                'green' if (self.t + self.phase_offsets[i]) % self.T_cycle < self.T_green
                else 'red' for i in range(self.N)}


# colour map for velocity: NaN -> white, 0 -> dark blue, v_max -> red
vel_cmap = plt.cm.RdYlGn  # green=fast, red=slow
print('Setup complete.')

## Animation 1: Free Flow vs Congestion

Cars are drawn as coloured rectangles along a 1-D road.
**Green = fast**, **red = slow/stopped**.

At low density the road stays mostly green; at high density it clogs with red/orange clusters.

In [ ]:
sims_1 = [
    NaSchOpen(L=L, v_max=V_MAX, p_rand=P_RAND, p_in=0.15, seed=1),
    NaSchOpen(L=L, v_max=V_MAX, p_rand=P_RAND, p_in=0.70, seed=2),
]
for s in sims_1:
    for _ in range(T_WARMUP): s.step()

fig1, axes1 = plt.subplots(2, 1, figsize=(14, 3.5))
labels1 = ['Free flow  ($p_{in} = 0.15$)', 'Congested  ($p_{in} = 0.70$)']

# represent road as a 1-row image of shape (1, L)
# NaN cells -> masked (white background)
road_imgs = []
for ax, sim, lbl in zip(axes1, sims_1, labels1):
    vg = sim.velocity_grid().reshape(1, -1)
    masked = np.ma.masked_invalid(vg)
    im = ax.imshow(masked, cmap=vel_cmap, vmin=0, vmax=1,
                   aspect='auto', interpolation='nearest', animated=True)
    ax.set_yticks([])
    ax.set_xlabel('Road position (cell)', fontsize=10)
    ax.set_title(lbl, fontsize=11)
    road_imgs.append(im)

fig1.suptitle('NaSch Road  (green = fast, red = slow)', fontsize=12)
step_lbl1 = fig1.text(0.5, 0.01, 'Step 0', ha='center', fontsize=10)
plt.tight_layout()

def update1(frame):
    for im, sim in zip(road_imgs, sims_1):
        sim.step()
        vg = sim.velocity_grid().reshape(1, -1)
        im.set_data(np.ma.masked_invalid(vg))
    step_lbl1.set_text(f'Step {T_WARMUP + frame + 1}')
    return road_imgs + [step_lbl1]

ani1 = animation.FuncAnimation(fig1, update1, frames=T_FRAMES,
                                interval=INTERVAL, blit=True)
plt.close()
HTML(ani1.to_jshtml())

## Animation 2: Stop-and-Go Waves

At high-but-not-saturated density, phantom jams form and travel **backwards** (upstream)
even as individual cars travel forwards.

Watch the red clusters: they drift to the **left** over time while cars move right through them.

In [ ]:
# Use periodic boundary to get a clean wave effect
class NaSchPeriodic:
    def __init__(self, L, n_cars, v_max=5, p_rand=0.3, seed=None):
        self.L      = L
        self.v_max  = v_max
        self.p_rand = p_rand
        rng = np.random.default_rng(seed)
        self.road = np.full(L, -1, dtype=int)
        pos = rng.choice(L, n_cars, replace=False)
        self.road[pos] = rng.integers(0, v_max + 1, n_cars)
        self.rng = rng

    def step(self):
        road, L, v_max, p_rand = self.road, self.L, self.v_max, self.p_rand
        car_pos = np.where(road >= 0)[0]
        if len(car_pos) == 0: return
        velocities = np.minimum(road[car_pos] + 1, v_max)
        gaps = np.empty(len(car_pos), dtype=int)
        for i, p in enumerate(car_pos):
            g = 0
            for j in range(1, L):
                if road[(p+j) % L] >= 0: break
                g += 1
            gaps[i] = g
        velocities = np.maximum(np.minimum(velocities, gaps), 0)
        mask = self.rng.random(len(car_pos)) < p_rand
        velocities[mask] = np.maximum(velocities[mask] - 1, 0)
        new_road = np.full(L, -1, dtype=int)
        new_road[(car_pos + velocities) % L] = velocities
        self.road = new_road

    def velocity_grid(self):
        g = np.full(self.L, np.nan)
        mask = self.road >= 0
        g[mask] = self.road[mask] / self.v_max
        return g


sim_wave = NaSchPeriodic(L=300, n_cars=120, v_max=V_MAX, p_rand=P_RAND, seed=3)
for _ in range(T_WARMUP): sim_wave.step()

fig2, ax2 = plt.subplots(figsize=(14, 1.8))
vg_w = sim_wave.velocity_grid().reshape(1, -1)
im_w = ax2.imshow(np.ma.masked_invalid(vg_w), cmap=vel_cmap, vmin=0, vmax=1,
                  aspect='auto', interpolation='nearest', animated=True)
ax2.set_yticks([])
ax2.set_xlabel('Road position (cell, periodic)', fontsize=10)
ax2.set_title('Stop-and-Go Waves  ($\\rho \\approx 0.40$, periodic, green=fast/red=stopped)', fontsize=11)
step_lbl2 = fig2.text(0.5, 0.01, 'Step 0', ha='center', fontsize=10)
plt.tight_layout()

def update2(frame):
    sim_wave.step()
    vg = sim_wave.velocity_grid().reshape(1, -1)
    im_w.set_data(np.ma.masked_invalid(vg))
    step_lbl2.set_text(f'Step {T_WARMUP + frame + 1}')
    return [im_w, step_lbl2]

ani2 = animation.FuncAnimation(fig2, update2, frames=T_FRAMES * 2,
                                interval=40, blit=True)
plt.close()
HTML(ani2.to_jshtml())

## Animation 3: Traffic Lights — Queue Discharge

Two traffic lights (vertical coloured bars: **green** / **red**) are placed at cells
$L/3$ and $2L/3$.  Watch cars queue up during the red phase, then discharge when it
turns green.

Compare **synchronised** lights (both red at the same time — double wall)
vs **offset** lights (green wave — cars pass through without stopping).

In [ ]:
sim_sync   = NaSchWithLights(N=2, T_cycle=40, T_green=20, offset=False,
                              L=L, v_max=V_MAX, p_rand=P_RAND, p_in=0.55, seed=4)
sim_offset = NaSchWithLights(N=2, T_cycle=40, T_green=20, offset=True,
                              L=L, v_max=V_MAX, p_rand=P_RAND, p_in=0.55, seed=5)
for s in [sim_sync, sim_offset]:
    for _ in range(T_WARMUP): s.step()

fig3, axes3 = plt.subplots(2, 1, figsize=(14, 4))
labels3 = ['Synchronised lights', 'Offset (green wave) lights']
sims3   = [sim_sync, sim_offset]

road_imgs3  = []
light_bars3 = []
for ax, sim, lbl in zip(axes3, sims3, labels3):
    vg   = sim.velocity_grid().reshape(1, -1)
    im   = ax.imshow(np.ma.masked_invalid(vg), cmap=vel_cmap, vmin=0, vmax=1,
                     aspect='auto', interpolation='nearest', animated=True)
    ax.set_yticks([])
    ax.set_xlabel('Road position (cell)', fontsize=10)
    ax.set_title(lbl, fontsize=11)
    # add light position markers (will be updated)
    bars = []
    for lp, state in sim.light_states().items():
        col = 'limegreen' if state == 'green' else 'red'
        bar = ax.axvline(lp, color=col, lw=3, alpha=0.85)
        bars.append((bar, lp))
    road_imgs3.append(im)
    light_bars3.append((bars, sim))

fig3.suptitle('NaSch with Traffic Lights  (green/red bars = light state)', fontsize=12)
step_lbl3 = fig3.text(0.5, 0.01, 'Step 0', ha='center', fontsize=10)
plt.tight_layout()

def update3(frame):
    artists = []
    for im, (bars, sim) in zip(road_imgs3, light_bars3):
        sim.step()
        vg = sim.velocity_grid().reshape(1, -1)
        im.set_data(np.ma.masked_invalid(vg))
        artists.append(im)
        states = sim.light_states()
        for bar, lp in bars:
            col = 'limegreen' if states.get(lp, 'green') == 'green' else 'red'
            bar.set_color(col)
            artists.append(bar)
    step_lbl3.set_text(f'Step {T_WARMUP + frame + 1}')
    artists.append(step_lbl3)
    return artists

ani3 = animation.FuncAnimation(fig3, update3, frames=T_FRAMES * 2,
                                interval=INTERVAL, blit=True)
plt.close()
HTML(ani3.to_jshtml())